In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

2025-06-05 14:11:20.908513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749132681.132927      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749132681.198152      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra/train.csv').drop(columns=['id'])
test = pandas.read_csv('/kaggle/input/zelestra/test.csv').drop(columns=['id'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]] = train[i[0]].fillna(0)
        #train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))
    else:
        test[i[0]] = test[i[0]].fillna(0)
        #test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
#train_x = train.drop(columns=["string_id","error_code","installation_type"])
train_x=train.drop(columns=['efficiency'])
train_y = train['efficiency']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [8]:
cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)

In [9]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

class HyperparameterTuner:
    def __init__(self, X_train, y_train, X_val, y_val, cat_features=None, cv_folds=5, n_trials=100):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.cat_features = cat_features
        self.cv_folds = cv_folds
        self.n_trials = n_trials
        self.best_params = {}
        
    def tune_catboost_main(self):
        """Tune CatBoost with Depthwise policy"""
        def objective(trial):
            # Bootstrap type selection with proper parameter handling
            bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS', 'No'])
            
            params = {
                'grow_policy': 'Depthwise',
                'iterations': trial.suggest_int('iterations', 800, 3000),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'depth': trial.suggest_int('depth', 6, 12),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10, log=True),
                'border_count': trial.suggest_categorical('border_count', [32, 64, 128, 254]),
                'max_ctr_complexity': trial.suggest_int('max_ctr_complexity', 1, 6),
                'bootstrap_type': bootstrap_type,
                'random_state': 42,
                'verbose': 0,
                'eval_metric': 'RMSE'
            }
            
            # Add bootstrap-specific parameters
            if bootstrap_type == 'Bayesian':
                params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.1, 1.0)
            elif bootstrap_type == 'Bernoulli':
                params['subsample'] = trial.suggest_float('subsample', 0.5, 0.9)
            elif bootstrap_type == 'MVS':
                params['subsample'] = trial.suggest_float('subsample_mvs', 0.8, 1.0)
            
            model = CatBoostRegressor(**params)
            
            # Cross-validation
            kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
            scores = []
            
            for train_idx, val_idx in kf.split(self.X_train):
                X_fold_train = self.X_train.iloc[train_idx]
                y_fold_train = self.y_train.iloc[train_idx]
                X_fold_val = self.X_train.iloc[val_idx]
                y_fold_val = self.y_train.iloc[val_idx]
                
                train_pool = Pool(X_fold_train, y_fold_train, cat_features=self.cat_features)
                val_pool = Pool(X_fold_val, y_fold_val, cat_features=self.cat_features)
                
                model.fit(train_pool, eval_set=val_pool, verbose=0, early_stopping_rounds=50)
                preds = model.predict(X_fold_val)
                rmse = np.sqrt(mean_squared_error(y_fold_val, preds))
                scores.append(rmse)
                
                # Pruning
                trial.report(rmse, len(scores))
                if trial.should_prune():
                    raise optuna.TrialPruned()
            
            return np.mean(scores)
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5)
        )
        
        print("Tuning CatBoost Main...")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        self.best_params['catboost_main'] = study.best_params
        print(f"Best CatBoost Main RMSE: {study.best_value:.4f}")
        
    def tune_catboost_deep(self):
        """Tune CatBoost with Lossguide policy"""
        def objective(trial):
            # Bootstrap type selection for Lossguide policy
            bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS', 'No'])
            
            params = {
                'grow_policy': 'Lossguide',
                'iterations': trial.suggest_int('iterations', 600, 2500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'depth': trial.suggest_int('depth', 8, 16),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 20, log=True),
                'max_leaves': trial.suggest_int('max_leaves', 16, 64),
                'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 10),
                'bootstrap_type': bootstrap_type,
                'random_state': 123,
                'verbose': 0,
                'eval_metric': 'RMSE'
            }
            
            # Add bootstrap-specific parameters
            if bootstrap_type == 'Bayesian':
                params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.1, 1.0)
            elif bootstrap_type == 'Bernoulli':
                params['subsample'] = trial.suggest_float('subsample', 0.5, 0.9)
            elif bootstrap_type == 'MVS':
                params['subsample'] = trial.suggest_float('subsample_mvs', 0.8, 1.0)
            
            model = CatBoostRegressor(**params)
            
            kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
            scores = []
            
            for train_idx, val_idx in kf.split(self.X_train):
                X_fold_train = self.X_train.iloc[train_idx]
                y_fold_train = self.y_train.iloc[train_idx]
                X_fold_val = self.X_train.iloc[val_idx]
                y_fold_val = self.y_train.iloc[val_idx]
                
                train_pool = Pool(X_fold_train, y_fold_train, cat_features=self.cat_features)
                val_pool = Pool(X_fold_val, y_fold_val, cat_features=self.cat_features)
                
                model.fit(train_pool, eval_set=val_pool, verbose=0, early_stopping_rounds=50)
                preds = model.predict(X_fold_val)
                rmse = np.sqrt(mean_squared_error(y_fold_val, preds))
                scores.append(rmse)
                
                trial.report(rmse, len(scores))
                if trial.should_prune():
                    raise optuna.TrialPruned()
            
            return np.mean(scores)
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5)
        )
        
        print("Tuning CatBoost Deep...")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        self.best_params['catboost_deep'] = study.best_params
        print(f"Best CatBoost Deep RMSE: {study.best_value:.4f}")
        
    def tune_xgboost(self):
        """Tune XGBoost"""
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10, log=True),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'random_state': 42,
                'n_jobs': -1
            }
            
            model = xgb.XGBRegressor(**params)
            
            kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
            scores = []
            
            for train_idx, val_idx in kf.split(self.X_train):
                X_fold_train = self.X_train.iloc[train_idx]
                y_fold_train = self.y_train.iloc[train_idx]
                X_fold_val = self.X_train.iloc[val_idx]
                y_fold_val = self.y_train.iloc[val_idx]
                
                model.fit(X_fold_train, y_fold_train, 
                         eval_set=[(X_fold_val, y_fold_val)], 
                         verbose=False, early_stopping_rounds=50)
                preds = model.predict(X_fold_val)
                rmse = np.sqrt(mean_squared_error(y_fold_val, preds))
                scores.append(rmse)
                
                trial.report(rmse, len(scores))
                if trial.should_prune():
                    raise optuna.TrialPruned()
            
            return np.mean(scores)
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5)
        )
        
        print("Tuning XGBoost...")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        self.best_params['xgboost'] = study.best_params
        print(f"Best XGBoost RMSE: {study.best_value:.4f}")
        
    def tune_lightgbm(self):
        """Tune LightGBM"""
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 16, 256),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
                'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
                'min_child_weight': trial.suggest_float('min_child_weight', 0.01, 10, log=True),
                'random_state': 42,
                'verbosity': -1
            }
            
            model = lgb.LGBMRegressor(**params)
            
            kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
            scores = []
            
            for train_idx, val_idx in kf.split(self.X_train):
                X_fold_train = self.X_train.iloc[train_idx]
                y_fold_train = self.y_train.iloc[train_idx]
                X_fold_val = self.X_train.iloc[val_idx]
                y_fold_val = self.y_train.iloc[val_idx]
                
                model.fit(X_fold_train, y_fold_train, 
                         eval_set=[(X_fold_val, y_fold_val)], 
                         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
                preds = model.predict(X_fold_val)
                rmse = np.sqrt(mean_squared_error(y_fold_val, preds))
                scores.append(rmse)
                
                trial.report(rmse, len(scores))
                if trial.should_prune():
                    raise optuna.TrialPruned()
            
            return np.mean(scores)
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5)
        )
        
        print("Tuning LightGBM...")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        self.best_params['lgbm'] = study.best_params
        print(f"Best LightGBM RMSE: {study.best_value:.4f}")
        
    def tune_extra_trees(self):
        """Tune Extra Trees"""
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
                'max_depth': trial.suggest_int('max_depth', 5, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None, 0.5, 0.7, 0.9]),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
                'random_state': 42,
                'n_jobs': -1
            }
            
            model = ExtraTreesRegressor(**params)
            scores = cross_val_score(model, self.X_train, self.y_train, 
                                   cv=self.cv_folds, scoring='neg_mean_squared_error')
            rmse = np.sqrt(-scores.mean())
            
            return rmse
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=5)
        )
        
        print("Tuning Extra Trees...")
        study.optimize(objective, n_trials=self.n_trials//2, show_progress_bar=True)
        self.best_params['extra_trees'] = study.best_params
        print(f"Best Extra Trees RMSE: {study.best_value:.4f}")
        
    def tune_neural_network(self):
        """Tune Neural Network"""
        def objective(trial):
            # Hidden layer architecture
            n_layers = trial.suggest_int('n_layers', 1, 4)
            hidden_layers = []
            for i in range(n_layers):
                layer_size = trial.suggest_int(f'layer_{i}_size', 50, 500)
                hidden_layers.append(layer_size)
            
            params = {
                'hidden_layer_sizes': tuple(hidden_layers),
                'activation': trial.suggest_categorical('activation', ['relu', 'tanh', 'logistic']),
                'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
                'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'invscaling', 'adaptive']),
                'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
                'max_iter': trial.suggest_int('max_iter', 200, 1000),
                'batch_size': trial.suggest_categorical('batch_size', ['auto', 32, 64, 128, 256]),
                'random_state': 42,
                'early_stopping': True,
                'validation_fraction': 0.1
            }
            
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(self.X_train.select_dtypes(include=[np.number]))
            
            model = MLPRegressor(**params)
            scores = cross_val_score(model, X_scaled, self.y_train, 
                                   cv=self.cv_folds, scoring='neg_mean_squared_error')
            rmse = np.sqrt(-scores.mean())
            
            return rmse
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42),
            pruner=MedianPruner(n_startup_trials=5)
        )
        
        print("Tuning Neural Network...")
        study.optimize(objective, n_trials=self.n_trials//3, show_progress_bar=True)
        self.best_params['neural_net'] = study.best_params
        print(f"Best Neural Network RMSE: {study.best_value:.4f}")
        
    def tune_meta_model(self):
        """Tune meta-model for stacking"""
        def objective(trial):
            model_type = trial.suggest_categorical('model_type', ['ridge', 'elastic_net', 'bayesian_ridge'])
            
            if model_type == 'ridge':
                params = {
                    'alpha': trial.suggest_float('alpha', 1e-3, 100, log=True),
                    'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False])
                }
                model = Ridge(**params)
            elif model_type == 'elastic_net':
                params = {
                    'alpha': trial.suggest_float('alpha', 1e-3, 10, log=True),
                    'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 0.9),
                    'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False])
                }
                model = ElasticNet(**params)
            else:  # bayesian_ridge
                params = {
                    'alpha_1': trial.suggest_float('alpha_1', 1e-8, 1e-3, log=True),
                    'alpha_2': trial.suggest_float('alpha_2', 1e-8, 1e-3, log=True),
                    'lambda_1': trial.suggest_float('lambda_1', 1e-8, 1e-3, log=True),
                    'lambda_2': trial.suggest_float('lambda_2', 1e-8, 1e-3, log=True)
                }
                model = BayesianRidge(**params)
            
            # Simple cross-validation for meta-model
            scores = cross_val_score(model, self.X_train, self.y_train, 
                                   cv=self.cv_folds, scoring='neg_mean_squared_error')
            rmse = np.sqrt(-scores.mean())
            
            return rmse
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42)
        )
        
        print("Tuning Meta Model...")
        study.optimize(objective, n_trials=50, show_progress_bar=True)
        self.best_params['meta_model'] = study.best_params
        print(f"Best Meta Model RMSE: {study.best_value:.4f}")
        
    def tune_all_models(self):
        """Tune all models in sequence"""
        print("Starting comprehensive hyperparameter tuning...")
        print("This may take a while depending on data size and n_trials...")
        
        # Tune base models
        self.tune_catboost_main()
        self.tune_catboost_deep()
        self.tune_xgboost()
        self.tune_lightgbm()
        self.tune_extra_trees()
        self.tune_neural_network()
        
        # Tune meta model
        self.tune_meta_model()
        
        print("\nHyperparameter tuning completed!")
        return self.best_params
    
    def create_tuned_models(self):
        """Create models with tuned hyperparameters"""
        if not self.best_params:
            raise ValueError("No tuned parameters found. Run tune_all_models() first.")
        
        tuned_models = {}
        
        # CatBoost Main
        cb_main_params = self.best_params['catboost_main'].copy()
        cb_main_params.update({
            'grow_policy': 'Depthwise',
            'random_state': 42,
            'verbose': 0,
            'eval_metric': 'RMSE'
        })
        tuned_models['catboost_main'] = CatBoostRegressor(**cb_main_params)
        
        # CatBoost Deep
        cb_deep_params = self.best_params['catboost_deep'].copy()
        cb_deep_params.update({
            'grow_policy': 'Lossguide',
            'random_state': 123,
            'verbose': 0,
            'eval_metric': 'RMSE'
        })
        tuned_models['catboost_deep'] = CatBoostRegressor(**cb_deep_params)
        
        # XGBoost
        xgb_params = self.best_params['xgboost'].copy()
        xgb_params.update({'random_state': 42, 'n_jobs': -1})
        tuned_models['xgboost'] = xgb.XGBRegressor(**xgb_params)
        
        # LightGBM
        lgb_params = self.best_params['lgbm'].copy()
        lgb_params.update({'random_state': 42, 'verbosity': -1})
        tuned_models['lgbm'] = lgb.LGBMRegressor(**lgb_params)
        
        # Extra Trees
        et_params = self.best_params['extra_trees'].copy()
        et_params.update({'random_state': 42, 'n_jobs': -1})
        tuned_models['extra_trees'] = ExtraTreesRegressor(**et_params)
        
        # Neural Network
        nn_params = self.best_params['neural_net'].copy()
        # Convert layer sizes back to tuple
        if 'n_layers' in nn_params:
            n_layers = nn_params.pop('n_layers')
            hidden_layers = []
            for i in range(n_layers):
                if f'layer_{i}_size' in nn_params:
                    hidden_layers.append(nn_params.pop(f'layer_{i}_size'))
            nn_params['hidden_layer_sizes'] = tuple(hidden_layers)
        
        nn_params.update({
            'random_state': 42,
            'early_stopping': True,
            'validation_fraction': 0.1
        })
        tuned_models['neural_net'] = MLPRegressor(**nn_params)
        
        return tuned_models
    
    def create_tuned_meta_model(self):
        """Create tuned meta-model"""
        if 'meta_model' not in self.best_params:
            return BayesianRidge()
        
        meta_params = self.best_params['meta_model'].copy()
        model_type = meta_params.pop('model_type', 'bayesian_ridge')
        
        if model_type == 'ridge':
            return Ridge(**meta_params)
        elif model_type == 'elastic_net':
            return ElasticNet(**meta_params)
        else:
            return BayesianRidge(**meta_params)


In [10]:
import json


tuner = HyperparameterTuner(
    X_train=X_train, 
    y_train=y_train, 
    X_val=X_test,  # Optional validation set
    y_val=y_test,
    cat_features=cat_features,
    cv_folds=5,
    n_trials=30  # Adjust based on time budget
)

# Run tuning (this takes time!)
#best_params = tuner.tune_all_models()
best_params={}
with open("/kaggle/input/zelestra-hyperparameters/best_hyperparameters.json", "r") as f:
    best_params=json.load(f)

In [11]:
# LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
#                'drop_rate': 0.8303473371870002,
#                'learning_rate': 0.2762739125344964,
#                'max_bin': 9983,
#                'max_depth': 8623,
#                'max_drop': 5480,
#                'min_child_samples': 101,
#                'min_data_in_leaf': 426,
#                'n_estimators': 1628,
#                'num_leaves': 3640,
#                'objective': 'regression_l1',
#                'reg_alpha': 0.39940189926691194,
#                'reg_lambda': 0.9567353299218986,
#                'skip_drop': 0.6274640597528743,
#                'verbosity': -1}

# XGB_R_parm = {'colsample_bytree': 0.871893537724492,
#               'gamma': 1,
#               'learning_rate': 0.923192518624813,
#               'max_depth': 15,
#               'min_child_weight': 1,
#               'n_estimators': 17748,
#               'objective': 'binary:logistic',
#               'reg_alpha': 45,
#               'reg_lambda': 0.8598696247943665,
#               'subsample': 0.9109367356405415}

# catboost_params = {'iterations' : 1000,
#                    'learning_rate': 0.009, 
#                    'depth': 7, 
#                    'l2_leaf_reg': 4, 
#                    'od_wait' : 50,
#                    'random_state' : 42,
#                    'eval_metric': 'RMSE', 
#                    'iterations': 1000,
#                    'bootstrap_type': 'Bayesian', 
#                    'allow_const_label': True, 
#                    'grow_policy' : 'Depthwise',
#                  }

In [12]:
# best_params['catboost_main']=catboost_params
# best_params['xgboost']=XGB_R_parm
# best_params['lgbm']=LGBM_R_parm

In [13]:
tuner.best_params=best_params

In [14]:
best_params['catboost_main'].keys()

dict_keys(['bootstrap_type', 'iterations', 'learning_rate', 'depth', 'l2_leaf_reg', 'border_count', 'max_ctr_complexity', 'subsample_mvs'])

In [15]:
best_params['catboost_main']['subsample']=best_params['catboost_main']['subsample_mvs']

In [16]:
del best_params['catboost_main']['subsample_mvs']

In [17]:
best_params['catboost_deep']['subsample']=best_params['catboost_deep']['subsample_mvs']

In [18]:
del best_params['catboost_deep']['subsample_mvs']

In [19]:
tuned_base_models = tuner.create_tuned_models()
tuned_meta_model = tuner.create_tuned_meta_model()

In [20]:
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

xgb.set_config(verbosity=0)

class SuperiorStackingRegressor:
    def __init__(self, base_models, meta_model=None, cv_folds=7, random_state=42):
        self.base_models = base_models
        self.meta_model = meta_model if meta_model else BayesianRidge()
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.fitted_models = []
        self.scaler = StandardScaler()
        self.optimal_weights = None
        self.base_preds_dict={}
        
    def _get_oof_predictions(self, X, y, cat_features=None):
        
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)

        oof_preds = np.zeros((X.shape[0], len(self.base_models)))
        
        print("Generating out-of-fold predictions with enhanced CV...")
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
            print(f"  Fold {fold + 1}/{self.cv_folds}")
            
            X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
            y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
            
            for i, (model_name, model) in enumerate(self.base_models.items()):
                model_copy = self._clone_model(model)
                
                if model_name == 'catboost_main':
                    train_pool = Pool(X_fold_train, y_fold_train, cat_features=cat_features)
                    val_pool = Pool(X_fold_val, y_fold_val, cat_features=cat_features)
                    model_copy.fit(train_pool, eval_set=val_pool, verbose=0)
                    oof_preds[val_idx, i] = model_copy.predict(X_fold_val)
                    
                elif model_name == 'catboost_deep':
                    train_pool = Pool(X_fold_train, y_fold_train, cat_features=cat_features)
                    val_pool = Pool(X_fold_val, y_fold_val, cat_features=cat_features)
                    model_copy.fit(train_pool, eval_set=val_pool, verbose=0)
                    oof_preds[val_idx, i] = model_copy.predict(X_fold_val)
                    
                elif model_name=='lgbm':
                    print('lgbm')
                    model_copy.fit(X_fold_train, y_fold_train, 
                                 eval_set=[(X_fold_val, y_fold_val)])
                    oof_preds[val_idx, i] = model_copy.predict(X_fold_val)
                elif model_name=='xgboost':
                    print('xgboost')
                    model_copy.fit(X_fold_train, y_fold_train, 
                                 eval_set=[(X_fold_val, y_fold_val)],verbose=False)
                    oof_preds[val_idx, i] = model_copy.predict(X_fold_val)
                elif model_name == 'neural_net':
                    # Scale data for neural network
                    scaler = StandardScaler()
                    X_scaled_train = scaler.fit_transform(X_fold_train.select_dtypes(include=[np.number]))
                    X_scaled_val = scaler.transform(X_fold_val.select_dtypes(include=[np.number]))
                    model_copy.fit(X_scaled_train, y_fold_train)
                    oof_preds[val_idx, i] = model_copy.predict(X_scaled_val)
                    
                else:  # Tree-based models
                    model_copy.fit(X_fold_train, y_fold_train)
                    oof_preds[val_idx, i] = model_copy.predict(X_fold_val)
        
        return oof_preds
    
    def _clone_model(self, model):
        """Create a copy of the model with same parameters"""
        return model.__class__(**model.get_params())
    
    def _optimize_weights_advanced(self, oof_preds, y_true):
        """Advanced weight optimization with regularization"""
        def objective(weights):
            weights = np.abs(weights)  # Ensure non-negative
            weights = weights / np.sum(weights)  # Normalize
            weighted_preds = np.average(oof_preds, axis=1, weights=weights)
            rmse = mp.sqrt(mean_squared_error(y_true, weighted_preds))
            score=100*(1-rmse)
            # Add regularization to prevent overfitting to one model
            regularization = 0.01 * np.sum(weights**2)
            return -score + regularization
        
        # Try multiple starting points
        best_weights = None
        best_score = float('inf')
        
        for seed in range(5):
            np.random.seed(seed)
            initial_weights = np.random.dirichlet(np.ones(len(self.base_models)))
            
            constraints = {'type': 'eq', 'fun': lambda w: np.sum(np.abs(w)) - 1}
            bounds = [(0, 1) for _ in range(len(self.base_models))]
            
            try:
                result = minimize(objective, initial_weights, method='SLSQP', 
                               bounds=bounds, constraints=constraints, 
                               options={'maxiter': 1000})
                print("Minimized")
                
                if result.success and result.fun < best_score:
                    best_score = result.fun
                    best_weights = np.abs(result.x) / np.sum(np.abs(result.x))
            except:
                continue
        
        return best_weights if best_weights is not None else np.ones(len(self.base_models)) / len(self.base_models)
    
    def fit(self, X, y, cat_features=None):
        """Fit the superior stacking regressor"""
        print("Training Superior Stacking Regressor...")
        
        oof_preds = self._get_oof_predictions(X, y, cat_features)
        
        print("Optimizing ensemble weights with advanced method...")
        self.optimal_weights = self._optimize_weights_advanced(oof_preds, y)
        
        print("Creating enhanced meta-features...")
        enhanced_features = self._create_enhanced_features(oof_preds)
        
        print("Training meta-model...")
        self.meta_model.fit(enhanced_features, y)
        
        print("Training final base models...")
        self.fitted_models = []
        
        for model_name, model in self.base_models.items():
            print(f"  Training {model_name}...")
            model_copy = self._clone_model(model)
            
            if 'catboost' in model_name:
                train_pool = Pool(X, y, cat_features=cat_features)
                model_copy.fit(train_pool, verbose=0)
                
            elif model_name in ['lgbm', 'xgboost']:
                model_copy.fit(X, y)
                
            elif model_name == 'neural_net':
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X.select_dtypes(include=[np.number]))
                model_copy.fit(X_scaled, y)
                self.nn_scaler = scaler  # Store scaler for predictions
            else:
                model_copy.fit(X, y)
            
            self.fitted_models.append((model_name, model_copy))
        
        # Print results
        self._print_results(oof_preds, y)
        
        return self
    
    def _create_enhanced_features(self, base_preds):
        """Create enhanced features for meta-model"""
        features = []
        
        # Original predictions
        features.append(base_preds)
        
        # Pairwise differences
        for i in range(base_preds.shape[1]):
            for j in range(i+1, base_preds.shape[1]):
                diff = (base_preds[:, i] - base_preds[:, j]).reshape(-1, 1)
                features.append(diff)
        
        # Statistics
        mean_pred = np.mean(base_preds, axis=1).reshape(-1, 1)
        std_pred = np.std(base_preds, axis=1).reshape(-1, 1)
        min_pred = np.min(base_preds, axis=1).reshape(-1, 1)
        max_pred = np.max(base_preds, axis=1).reshape(-1, 1)
        
        features.extend([mean_pred, std_pred, min_pred, max_pred])
        
        # Combine all features
        enhanced = np.hstack(features)
        
        # Scale features
        return self.scaler.fit_transform(enhanced)
    
    def predict(self, X):
        """Make predictions using the trained ensemble"""
        # Get base model predictions
        base_preds = []
        base_preds_dict={}
        for model_name, model in self.fitted_models:
            if 'catboost' in model_name:
                preds = model.predict(X)
            elif model_name == 'neural_net':
                X_scaled = self.nn_scaler.transform(X.select_dtypes(include=[np.number]))
                preds = model.predict(X_scaled)
            else:
                preds = model.predict(X)
            self.base_preds_dict[model_name]=preds
            base_preds.append(preds)
        
        base_preds = np.column_stack(base_preds)
        
        # Create enhanced features
        enhanced_features = self._create_enhanced_features_predict(base_preds)
        
        # Get meta-model predictions
        meta_preds = self.meta_model.predict(enhanced_features)
        
        # Get weighted average predictions
        weighted_preds = np.average(base_preds, axis=1, weights=self.optimal_weights)
        
        # Advanced combination strategy
        final_preds = 0.7 * meta_preds + 0.3 * weighted_preds
        
        return final_preds
    
    def _create_enhanced_features_predict(self, base_preds):
        """Create enhanced features for prediction"""
        features = []
        
        # Original predictions
        features.append(base_preds)
        
        # Pairwise differences
        for i in range(base_preds.shape[1]):
            for j in range(i+1, base_preds.shape[1]):
                diff = (base_preds[:, i] - base_preds[:, j]).reshape(-1, 1)
                features.append(diff)
        
        # Statistics
        mean_pred = np.mean(base_preds, axis=1).reshape(-1, 1)
        std_pred = np.std(base_preds, axis=1).reshape(-1, 1)
        min_pred = np.min(base_preds, axis=1).reshape(-1, 1)
        max_pred = np.max(base_preds, axis=1).reshape(-1, 1)
        
        features.extend([mean_pred, std_pred, min_pred, max_pred])
        
        # Combine and scale
        enhanced = np.hstack(features)
        return self.scaler.transform(enhanced)
    
    def _print_results(self, oof_preds, y_true):
        """Print detailed results"""
        print(f"\nOptimal weights:")
        for i, (model_name, _) in enumerate(self.base_models.items()):
            print(f"  {model_name}: {self.optimal_weights[i]:.4f}")
        
        print(f"\nOut-of-fold RMSE scores:")
        for i, (model_name, _) in enumerate(self.base_models.items()):
            rmse = np.sqrt(mean_squared_error(y_true, oof_preds[:, i]))
            print(f"  {model_name}: {rmse:.4f}")
        
        # Weighted ensemble OOF score
        weighted_oof = np.average(oof_preds, axis=1, weights=self.optimal_weights)
        weighted_rmse = np.sqrt(mean_squared_error(y_true, weighted_oof))
        print(f"  Weighted Ensemble: {weighted_rmse:.4f}")


In [21]:
print("Training Superior Stacking Regressor...")
stacker = SuperiorStackingRegressor(
    base_models=tuned_base_models,
    meta_model=tuned_meta_model,
    cv_folds=30,
    random_state=30
)


cat_features = list(X_train.select_dtypes(include=['object', 'category']).columns)

# Fit the stacking regressor
stacker.fit(X_train, y_train, cat_features=cat_features)

# Make predictions
ensemble_preds = stacker.predict(X_test)

# Calculate and compare results
ensemble_rmse = np.sqrt(mean_squared_error(y_test, ensemble_preds))

Training Superior Stacking Regressor...
Training Superior Stacking Regressor...
Generating out-of-fold predictions with enhanced CV...
  Fold 1/30
xgboost
lgbm
  Fold 2/30
xgboost
lgbm
  Fold 3/30
xgboost
lgbm
  Fold 4/30
xgboost
lgbm
  Fold 5/30
xgboost
lgbm
  Fold 6/30
xgboost
lgbm
  Fold 7/30
xgboost
lgbm
  Fold 8/30
xgboost
lgbm
  Fold 9/30
xgboost
lgbm
  Fold 10/30
xgboost
lgbm
  Fold 11/30
xgboost
lgbm
  Fold 12/30
xgboost
lgbm
  Fold 13/30
xgboost
lgbm
  Fold 14/30
xgboost
lgbm
  Fold 15/30
xgboost
lgbm
  Fold 16/30
xgboost
lgbm
  Fold 17/30
xgboost
lgbm
  Fold 18/30
xgboost
lgbm
  Fold 19/30
xgboost
lgbm
  Fold 20/30
xgboost
lgbm
  Fold 21/30
xgboost
lgbm
  Fold 22/30
xgboost
lgbm
  Fold 23/30
xgboost
lgbm
  Fold 24/30
xgboost
lgbm
  Fold 25/30
xgboost
lgbm
  Fold 26/30
xgboost
lgbm
  Fold 27/30
xgboost
lgbm
  Fold 28/30
xgboost
lgbm
  Fold 29/30
xgboost
lgbm
  Fold 30/30
xgboost
lgbm
Optimizing ensemble weights with advanced method...
Creating enhanced meta-features...
Trainin

In [22]:
print(100*(1-ensemble_rmse))

90.25176864885341


In [23]:
stacker.base_preds_dict

{'catboost_main': array([0.57927509, 0.43169156, 0.44812685, ..., 0.48838198, 0.37237796,
        0.3692506 ]),
 'catboost_deep': array([0.58733202, 0.42148805, 0.44904956, ..., 0.5001205 , 0.36791466,
        0.3883292 ]),
 'xgboost': array([0.5827704 , 0.4424336 , 0.45176816, ..., 0.49501824, 0.37070203,
        0.38068116], dtype=float32),
 'lgbm': array([0.58803592, 0.430931  , 0.4430847 , ..., 0.49446609, 0.37214758,
        0.38299886]),
 'extra_trees': array([0.59409792, 0.42748485, 0.44233818, ..., 0.49547384, 0.36281728,
        0.39782139]),
 'neural_net': array([0.54667492, 0.41297202, 0.44091153, ..., 0.49310748, 0.4054079 ,
        0.37367084])}

In [24]:
ensemble_preds = stacker.predict(test)

In [25]:
stacker.base_preds_dict

{'catboost_main': array([0.4001109 , 0.54603146, 0.51796155, ..., 0.63784553, 0.43685615,
        0.56753802]),
 'catboost_deep': array([0.3996617 , 0.54153041, 0.51890809, ..., 0.62661169, 0.45680111,
        0.56837106]),
 'xgboost': array([0.40643305, 0.537299  , 0.5117498 , ..., 0.6349501 , 0.4373265 ,
        0.5615711 ], dtype=float32),
 'lgbm': array([0.4100912 , 0.55404127, 0.51568011, ..., 0.61799224, 0.44361244,
        0.56001741]),
 'extra_trees': array([0.40993039, 0.53953263, 0.52541289, ..., 0.61682683, 0.43654042,
        0.54914612]),
 'neural_net': array([0.4116072 , 0.53063069, 0.49933753, ..., 0.57313101, 0.44846406,
        0.5469914 ])}

In [26]:
# single_preds = enhanced_cat.predict(test)

In [27]:
# cat_preds = stacker.fitted_models[0][1].predict(test)
# rf_preds = stacker.fitted_models[1][1].predict(test)
# lgb_preds = stacker.fitted_models[2][1].predict(test)
# simple_avg = (cat_preds + rf_preds + lgb_preds) / 3

In [28]:
# final=cat_pre_test+rf_pre_test+lgb_pre_test
# final/=3.0

In [29]:
id = pandas.read_csv('/kaggle/input/zelestra/test.csv')['id']
test_predictions = (ensemble_preds)
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.4046139 , 0.54831806, 0.51825685, ..., 0.62670828, 0.44142653,
       0.56290454])

In [30]:
id = pandas.read_csv('/kaggle/input/zelestra/test.csv')['id']
test_predictions = (stacker.base_preds_dict['catboost_deep'])
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.3996617 , 0.54153041, 0.51890809, ..., 0.62661169, 0.45680111,
       0.56837106])

In [31]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.399662
1,1,0.541530
2,2,0.518908
3,3,0.461556
4,4,0.454743
...,...,...
11995,11995,0.549968
11996,11996,0.473876
11997,11997,0.626612
11998,11998,0.456801


In [32]:
submission.to_csv('catboostdeep2.csv', index = False)